### Turkish Music Emotion

La música desempeña un papel fundamental en la vida humana, ya que tiene la capacidad de despertar o transmitir sentimientos. Dado que el reconocimiento de emociones musicales es objeto de numerosos estudios en diversas disciplinas, como la ciencia, la psicología, la musicología y el arte, ha atraído la atención de los investigadores como un tema de investigación de actualidad en los últimos años. Muchos investigadores extraen características acústicas de la música e investigan las relaciones entre las etiquetas emocionales correspondientes a estas características. Por otro lado, en estudios recientes, los tipos de música se clasifican emocionalmente mediante aprendizaje profundo a través de espectrogramas musicales que incluyen información tanto del dominio temporal como del frecuencial. En el presente estudio, se presenta un nuevo método para el reconocimiento de emociones musicales mediante un modelo de aprendizaje profundo preentrenado con espectrogramas de croma extraídos de grabaciones musicales. Se utiliza la arquitectura AlexNet como modelo de red preentrenado. Las capas conv5, Fc6, Fc7 y Fc8 del modelo AlexNet se seleccionan como capa de extracción de características, y de estas capas se extraen características visuales profundas. Las características profundas extraídas se utilizan para entrenar y probar las Máquinas de Vectores de Soporte (SVM) y los clasificadores Softmax. Además, se extraen características visuales profundas de las capas conv5_3, Fc6, Fc7 y Fc8 del modelo de red profunda VGG-16 y se realizan las mismas aplicaciones experimentales para determinar la eficacia de las redes profundas preentrenadas en el reconocimiento de emociones musicales. Se realizaron varios experimentos con dos conjuntos de datos y se obtuvieron mejores resultados con el método propuesto. El mejor resultado se obtuvo con VGG-16 en la capa Fc7, con un 89,2 % en nuestro conjunto de datos. Según los resultados obtenidos, se observa que el método presentado ofrece un mejor rendimiento.

In [10]:
# Importar las librerias

import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from datetime import datetime


In [11]:
# Inicializar MLFlow Server

mlflow.set_tracking_uri("http://127.0.0.1:8080")
mlflow.set_experiment("01 - Turkish Music Emotion")
mlflow.sklearn.autolog()

In [12]:
# Cargar Turkish Music Emotion Dataset

# Cargar el dataset
df = pd.read_csv("../datasets/input/turkish_music_emotion_original.csv")

print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())

print("\nClass distribution:")
print(df['Class'].value_counts())

print("\nDataset info:")
print(df.info())

Dataset shape: (400, 51)

First few rows:
   Class  _RMSenergy_Mean  _Lowenergy_Mean  _Fluctuation_Mean  _Tempo_Mean  \
0  relax            0.052            0.591              9.136      130.043   
1  relax            0.125            0.439              6.680      142.240   
2  relax            0.046            0.639             10.578      188.154   
3  relax            0.135            0.603             10.442       65.991   
4  relax            0.066            0.591              9.769       88.890   

   _MFCC_Mean_1  _MFCC_Mean_2  _MFCC_Mean_3  _MFCC_Mean_4  _MFCC_Mean_5  ...  \
0         3.997         0.363         0.887         0.078         0.221  ...   
1         4.058         0.516         0.785         0.397         0.556  ...   
2         2.775         0.903         0.502         0.329         0.287  ...   
3         2.841         1.552         0.612         0.351         0.011  ...   
4         3.217         0.228         0.814         0.096         0.434  ...   

   _Chro

In [13]:
# Data Preprocessing y Model Training

# Separar features y target
X = df.drop('Class', axis=1)
y = df['Class']

# Encode la variable target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Separar los datos en training y test
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

# Crear nombre único con timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_name = f"turkish_music_emotion_{timestamp}"

print(f"Starting run: {run_name}")

# Entrenar y evaluar el modelo
with mlflow.start_run(run_name=run_name):
    # Entrenar el modelo Random Forest
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    
    # Predicciones
    y_pred = model.predict(X_test)
    
    # Calcular accuracy
    accuracy = accuracy_score(y_test, y_pred)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("timestamp", timestamp)
    
    print(f"Model Accuracy: {accuracy:.4f}")
    print(f"Run Name: {run_name}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    
    # Log the model
    mlflow.sklearn.log_model(model, "model")

Training set size: 320
Test set size: 80
Number of features: 50
Starting run: turkish_music_emotion_20251001_115928


2025/10/01 11:59:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model Accuracy: 0.7625
Run Name: turkish_music_emotion_20251001_115928

Classification Report:
              precision    recall  f1-score   support

       angry       0.88      0.75      0.81        20
       happy       0.80      1.00      0.89        20
       relax       0.68      0.75      0.71        20
         sad       0.69      0.55      0.61        20

    accuracy                           0.76        80
   macro avg       0.76      0.76      0.76        80
weighted avg       0.76      0.76      0.76        80



2025/10/01 11:59:33 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run turkish_music_emotion_20251001_115928 at: http://127.0.0.1:8080/#/experiments/200394103665423037/runs/68827f41ad804dd4830fec8121860a8b
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/200394103665423037
